In [1]:


from collections import Counter
import os
import shutil
import random
from pathlib import Path

1️. Counting images in my dataset

In [2]:
DATA_ROOT = r"C:\Users\HP\Desktop\pneumonia_ai_v1\data\chest_xray"

def count_images(root):
    import os
    from collections import Counter
    counts = Counter()
    for split in ['train', 'val', 'test']:
        for label in ['NORMAL', 'PNEUMONIA']:
            path = os.path.join(root, split, label)
            counts[f"{split}_{label}"] = len([
                f for f in os.listdir(path)
                if f.lower().endswith(('.jpeg', '.jpg', '.png'))
            ])
    return counts

print(count_images(DATA_ROOT))


Counter({'train_PNEUMONIA': 3875, 'train_NORMAL': 1341, 'test_PNEUMONIA': 390, 'test_NORMAL': 234, 'val_NORMAL': 8, 'val_PNEUMONIA': 8})


2️. Moving 20% of the train images to create a better validation set

In [5]:


# Set seed for reproducibility
random.seed(42)

# Define paths 
project_root = Path(r"C:\Users\HP\Desktop\pneumonia_ai_v1\data")
data_root = project_root /"chest_xray"
train_dir = data_root / "train"
val_dir = data_root / "val"

def safe_stratified_split(frac=0.2):
    """
    Moves 20% of images from train to val ONLY if val is nearly empty.
    This prevents double-moving files if the script is run twice.
    """
    for cls in ["NORMAL", "PNEUMONIA"]:
        src_path = train_dir / cls
        dst_path = val_dir / cls
        
        # Ensure destination exists
        dst_path.mkdir(parents=True, exist_ok=True)
        
        # Get list of files
        train_files = [f for f in src_path.iterdir() if f.is_file()]
        val_files = [f for f in dst_path.iterdir() if f.is_file()]
        
        # Logic: Only move if the val folder has very few files (like the original 8)
        # and train has many. This is a safety check.
        if len(val_files) < 20: 
            print(f"Splitting {cls}... (Current train: {len(train_files)})")
            
            random.shuffle(train_files)
            num_to_move = int(len(train_files) * frac)
            files_to_move = train_files[:num_to_move]
            
            for f in files_to_move:
                shutil.move(str(f), dst_path / f.name)
            
            print(f"Successfully moved {num_to_move} images to {cls} validation.")
        else:
            print(f"Split already exists for {cls}. Validation has {len(val_files)} images. Skipping.")

# Run the split
safe_stratified_split(frac=0.2)

Splitting NORMAL... (Current train: 1342)
Successfully moved 268 images to NORMAL validation.
Splitting PNEUMONIA... (Current train: 3876)
Successfully moved 775 images to PNEUMONIA validation.


3️. Verifyign the new counts

In [6]:


def count_images(root):
    counts = Counter()
    for split in ['train', 'val', 'test']:
        for label in ['NORMAL', 'PNEUMONIA']:
            path = os.path.join(root, split, label)
            counts[f"{split}_{label}"] = len([
                f for f in os.listdir(path)
                if f.lower().endswith(('.jpeg', '.jpg', '.png'))
            ])
    return counts

DATA_ROOT = r"C:\Users\HP\Desktop\pneumonia_ai_v1\data\chest_xray"
print(count_images(DATA_ROOT))


Counter({'train_PNEUMONIA': 3101, 'train_NORMAL': 1073, 'val_PNEUMONIA': 782, 'test_PNEUMONIA': 390, 'val_NORMAL': 276, 'test_NORMAL': 234})
